# Do Enem à graduação: minha parte de dados e gráficos

**Brasil · 2020 a 2024 · Pandas e Plotly**

Esta entrega ocupa somente `data/` e `notebooks/`, conforme a divisão de tarefas do grupo.
O notebook contém o código de leitura, exploração e quatro gráficos, sem importar módulos de `src/` ou `scripts/`.
As tabelas oficiais tratadas acompanham o notebook; os arquivos brutos ficam fora do Git.

**Perguntas:** como evoluíram as inscrições dos concluintes da rede pública no Enem?
Como se distribuem as redes registradas e os registros sem informação?
Como a série se apresenta ao lado do ingresso na graduação?

**Limite:** inscrições no Enem não identificam ingresso pelo Enem. As séries agregadas não acompanham as mesmas pessoas.
O roteiro e as limitações dos dados continuam explicados nas próximas seções.

## 1. Ambiente e arquivos

Abra este notebook na raiz do projeto ou em `notebooks/`. Ele procura `data/processed/` automaticamente.
Dependências: **pandas, plotly e openpyxl**; use um kernel Python no VS Code ou Jupyter.
Se precisar instalar, execute `%pip install pandas plotly openpyxl` em uma célula do seu ambiente.
Para repetir o download opcional dos originais, também é necessário `truststore`.

Não é preciso alterar o `requirements.txt` do grupo para entregar esta contribuição.
Os resultados e as quatro figuras já estão salvos no notebook.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'data/processed/enem_brasil_rede.csv').exists()), None)
if ROOT is None:
    raise RuntimeError('Coloque data/ e notebooks/ na raiz do mesmo projeto.')
PROCESSED = ROOT / 'data/processed'
print('Pasta de dados encontrada: data/processed/')
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

Pasta de dados encontrada: data/processed/


### Código de apoio dentro do notebook

Estas classes fazem a leitura das tabelas, os cálculos exploratórios e as figuras.
Elas ficam nesta célula para que minha entrega funcione apenas com as duas pastas combinadas.
A equipe pode aproveitar esse código na aplicação depois, sem precisar modificar arquivos dos colegas agora.

In [2]:
"""Leitura das fontes: o mesmo contrato para CSV e planilhas do Inep."""
from abc import ABC, abstractmethod
from pathlib import Path

import pandas as pd
from openpyxl import load_workbook


class FonteDados(ABC):
    def __init__(self, caminho):
        self._caminho = Path(caminho)

    @property
    def caminho(self):
        return self._caminho

    @abstractmethod
    def carregar(self):
        """Retorna um DataFrame sem alterar a fonte."""


class FonteCSV(FonteDados):
    def carregar(self):
        return pd.read_csv(self.caminho, encoding='utf-8')


class FonteExcel(FonteDados):
    def __init__(self, caminho, aba, limite_linhas=None):
        super().__init__(caminho)
        self._aba = aba
        self._limite_linhas = limite_linhas

    def carregar(self):
        # As sinopses possuem milhares de municípios. Para o Brasil, basta ler
        # o cabeçalho e a linha nacional, sem carregar o restante na memória.
        livro = load_workbook(self.caminho, read_only=True, data_only=True)
        try:
            if self._aba not in livro.sheetnames:
                raise ValueError(f'Aba ausente: {self._aba} em {self.caminho.name}')
            return pd.DataFrame(livro[self._aba].iter_rows(
                max_row=self._limite_linhas, values_only=True))
        finally:
            livro.close()

"""Indicadores descritivos para exploração; não calculam aprovação/transição."""
import pandas as pd


class AnalisadorColeta:
    def __init__(self, enem):
        self._enem = enem.copy()

    def resumo(self):
        tabela = self._enem.pivot(index='ano', columns='rede_origem', values='inscritos_concluintes')
        tabela = tabela.rename(columns={'Pública':'inscritos_publica', 'Privada':'inscritos_privada',
                                         'Não dispõe da informação':'inscritos_sem_informacao'})
        tabela['total_concluintes_inscritos'] = tabela.sum(axis=1, min_count=3)
        total = tabela['total_concluintes_inscritos'].where(tabela['total_concluintes_inscritos'].ne(0))
        tabela['participacao_publica_no_total_pct'] = tabela.inscritos_publica / total * 100
        tabela['sem_informacao_pct'] = tabela.inscritos_sem_informacao / total * 100
        tabela['variacao_anual_publica_pct'] = tabela.inscritos_publica.pct_change(fill_method=None) * 100
        return tabela.sort_index().reset_index().rename_axis(None, axis=1)

"""Figuras Plotly reutilizáveis no notebook e, posteriormente, no dashboard."""
from pathlib import Path
import plotly.express as px

CORES = {'Pública':'#147d92', 'Privada':'#e68a36', 'Não dispõe da informação':'#9aa6b2'}
AVISO = ('Rede registrada na sinopse; sem informação preservada. '
         '2024 tem forte mudança de cobertura da categoria privada.')


class Visualizador:
    def __init__(self):
        self._cores = CORES.copy()

    def _formatar(self, figura, titulo, subtitulo, fonte):
        figura.update_layout(
            template='plotly_white', height=580,
            title=dict(text=f'{titulo}<br><sup>{subtitulo}</sup>', font=dict(size=21)),
            font=dict(family='Arial', size=13, color='#23334a'),
            margin=dict(l=85, r=40, t=110, b=150),
            legend_title_text='Rede da escola de origem',
        )
        figura.add_annotation(x=0, y=-0.24, xref='paper', yref='paper', showarrow=False,
                              xanchor='left', align='left',
                              text=fonte+'<br>'+AVISO, font=dict(size=11))
        return figura

    def linhas(self, enem):
        publica = enem.loc[enem.rede_origem.eq('Pública')].sort_values('ano')
        fig = px.line(publica, x='ano', y='inscritos_concluintes', markers=True,
                      color_discrete_sequence=[self._cores['Pública']],
                      labels={'ano':'Edição do Enem', 'inscritos_concluintes':'Inscrições de concluintes'})
        fig.update_xaxes(dtick=1)
        fig.update_yaxes(rangemode='tozero')
        return self._formatar(fig, 'Concluintes da rede pública inscritos no Enem',
                             'Brasil · 2020–2024 · contagem de inscrições, não aprovações',
                             'Fonte: Inep, Sinopses do Enem, tabela 3.10 de cada edição.')

    def barras(self, enem):
        dados = enem[enem.rede_origem.isin(['Pública', 'Privada'])]
        fig = px.bar(dados, x='ano', y='inscritos_concluintes', color='rede_origem',
                     barmode='group', color_discrete_map=self._cores,
                     labels={'ano':'Edição do Enem', 'inscritos_concluintes':'Inscrições de concluintes',
                             'rede_origem':'Rede de origem'})
        fig.update_xaxes(dtick=1)
        return self._formatar(fig, 'Inscrições por rede registrada',
                             'Brasil · 2020–2024 · barras não representam taxas de acesso',
                             'Fonte: Inep, Sinopses do Enem, tabela 3.10. Categoria sem informação no gráfico de setores.')

    def setores(self, enem):
        ano = int(enem.ano.max())
        dados = enem[enem.ano.eq(ano)]
        fig = px.pie(dados, names='rede_origem', values='inscritos_concluintes',
                     color='rede_origem', color_discrete_map=self._cores, hole=0.45)
        fig.update_traces(textinfo='percent', textposition='outside',
                          hovertemplate='%{label}<br>%{value:,.0f} inscrições<br>%{percent}<extra></extra>')
        return self._formatar(fig, f'Composição das inscrições de concluintes · {ano}',
                             'Brasil · inclui registros sem informação da rede',
                             f'Fonte: Inep, Sinopse do Enem {ano}, tabela 3.10. Denominador: todos os concluintes inscritos.')

    def dispersao(self, enem, superior):
        publica = enem[enem.rede_origem.eq('Pública')]
        dados = publica.merge(superior, on='ano', validate='one_to_one')
        fig = px.scatter(dados, x='inscritos_concluintes', y='ingressantes_graduacao', text='ano',
                         color_discrete_sequence=[self._cores['Pública']],
                         labels={'inscritos_concluintes':'Inscrições de concluintes da rede pública no Enem',
                                 'ingressantes_graduacao':'Ingressantes na graduação · todas as IES e formas'})
        posicoes = {2020:'bottom right', 2021:'top left', 2022:'bottom right',
                    2023:'top left', 2024:'top center'}
        fig.update_traces(marker_size=14,
                          textposition=[posicoes.get(int(ano), 'top center') for ano in dados.ano])
        fig.update_xaxes(rangemode='tozero')
        fig.update_yaxes(rangemode='tozero')
        return self._formatar(fig, 'Enem e ingresso na graduação: séries agregadas',
                             'Brasil · 2020–2024 · cinco anos; sem vínculo individual ou inferência causal',
                             'Fonte: Inep, Enem 3.10 e Educação Superior 4.4. Mesmos anos, populações distintas.')

    def todas(self, enem, superior):
        return {'01_linhas':self.linhas(enem), '02_barras':self.barras(enem),
                '03_setores':self.setores(enem), '04_dispersao':self.dispersao(enem, superior)}

    def exportar(self, figuras, pasta):
        pasta = Path(pasta)
        pasta.mkdir(parents=True, exist_ok=True)
        for nome, figura in figuras.items():
            figura.write_html(pasta/f'{nome}.html', include_plotlyjs='directory',
                              config={'responsive':True, 'displaylogo':False})
        return pasta

## 2. Catálogo das fontes e cobertura

Usamos 15 sinopses anuais (cinco edições de cada uma das três bases) e um arquivo de trajetória da coorte 2020.
O catálogo traz links, abas, datas operacionais e hashes dos arquivos efetivamente processados.
Os painéis do Inep ajudam a consultar os dados, mas esta entrega extrai e confere os números nas planilhas.

| Base | Tabela | O que mede | Uso |
|---|---|---|---|
| Enem | 3.10 | Inscritos que cursavam o último ano e concluiriam naquele ano, por dependência administrativa | Gráficos principais |
| Censo Escolar | 1.25 | Matrículas em todas as séries do ensino médio regular | Contexto, sem usar como concluintes |
| Censo Superior | 4.4 | Ingressantes na graduação, presencial e EaD, todas as formas | Comparação descritiva |
| Trajetória | Coorte 2020, referências 2020–2024 | Vínculos com cursos, permanência e eventos anuais | Contexto e exploração |

Não foram necessários microdados do Enem: as cinco sinopses possuem a tabela 3.10 exigida para os gráficos.
O Enem 2020 foi aplicado em 2021; os anos deste notebook seguem a **edição** do exame.

In [3]:
fontes = pd.DataFrame(json.loads((PROCESSED / 'fontes.json').read_text(encoding='utf-8')))
display(fontes[['id', 'ano', 'aba', 'pagina', 'url', 'data_arquivo_local', 'sha256_zip']])
assert len(fontes) == 16
assert fontes['id'].is_unique

,id,ano,aba,pagina,url,data_arquivo_local,sha256_zip
0,enem_2020,2020,3.10,https://www.gov.br/inep/pt-br/acesso-a-informa...,https://download.inep.gov.br/informacoes_estat...,2026-11-19T20:13:20.949470-03:00,976f4590904d3c582146cd57870774255ce468cbf0ea4d...
1,superior_2020,2020,4.4,https://www.gov.br/inep/pt-br/acesso-a-informa...,https://download.inep.gov.br/informacoes_estat...,2026-11-19T20:21:23.060462-03:00,6172df8ef2591a1467a73f9d67e3a7b0d027ff286da49f...
2,basica_2020,2020,Ensino Médio 1.25,https://www.gov.br/inep/pt-br/acesso-a-informa...,https://download.inep.gov.br/dados_abertos/sin...,2026-11-19T20:20:34.741928-03:00,a252b0e5e8668503b54dbc3ef8c91f67d99bca042e872c...
3,enem_2021,2021,3.10,https://www.gov.br/inep/pt-br/acesso-a-informa...,https://download.inep.gov.br/informacoes_estat...,2026-11-19T20:18:08.169716-03:00,5fa5d9efa3232cadde903c13fdac22fdb87ce11ec6a4c8...
4,superior_2021,2021,4.4,https://www.gov.br/inep/pt-br/acesso-a-informa...,https://download.inep.gov.br/informacoes_estat...,2026-11-19T20:23:08.904004-03:00,0483c5677331eaa2724ceddc064ccfbe7c94b3f7c791b0...
5,basica_2021,2021,Ensino Médio 1.25,https://www.gov.br/inep/pt-br/acesso-a-informa...,https://download.inep.gov.br/dados_abertos/sin...,2026-11-19T20:21:16.805187-03:00,66861b1e0adc7e09e503ca3683f960f644707a69be7c6a...
6,enem_2022,2022,3.10,https://www.gov.br/inep/pt-br/acesso-a-informa...,https://download.inep.gov.br/informacoes_estat...,2026-11-19T20:18:43.963508-03:00,3f8c3b7d696bc1f6f5d0b421f3b1a18e7f911c844a21ba...
7,superior_2022,2022,4.4,https://www.gov.br/inep/pt-br/acesso-a-informa...,https://download.inep.gov.br/informacoes_estat...,2026-11-19T20:25:08.639771-03:00,6ce329f291618d0a68d2bb8e48a49cdeb0d383b5691e12...
8,basica_2022,2022,Ensino Médio 1.25,https://www.gov.br/inep/pt-br/acesso-a-informa...,https://download.inep.gov.br/dados_abertos/sin...,2026-11-19T20:23:01.109022-03:00,811d291a9ad716906132e162e871c8e0f8fc09b092205a...
9,enem_2023,2023,3.10,https://www.gov.br/inep/pt-br/acesso-a-informa...,https://download.inep.gov.br/informacoes_estat...,2026-11-19T20:18:45.653694-03:00,e6b7bfa46694966c23bb2ac5e5b8eb504065d8b99c9b53...


## 3. Conferência de uma amostra original

As sinopses têm títulos e cabeçalhos em várias linhas. Não usamos `skiprows=8` indiscriminadamente.
O extrator verifica o título, o ano e os campos, encontra a linha **Brasil** e deixa os subtotais de regiões e UFs de fora.
As células da linha nacional foram preservadas em `evidencias_extracao.json` para conferência mesmo sem os arquivos grandes.

Na tabela do Enem, **Pública = Federal + Estadual + Municipal**. A categoria sem informação não é atribuída a nenhuma rede.

In [4]:
evidencias = json.loads((PROCESSED / 'evidencias_extracao.json').read_text(encoding='utf-8'))
exemplo = next(e for e in evidencias if e['ano'] == 2020 and e['aba'] == '3.10')
print(exemplo['titulo'])
print('Linha nacional no Excel:', exemplo['linha_excel'])
display(pd.DataFrame(exemplo['celulas'].items(), columns=['celula', 'valor_original']))

# Se os brutos estiverem disponíveis, mostramos também a leitura direta da fonte.
fonte_2020 = fontes.loc[fontes.id.eq('enem_2020')].iloc[0]
caminho_bruto = ROOT / fonte_2020.arquivo_xlsx
if caminho_bruto.exists():
    amostra_original = FonteExcel(caminho_bruto, '3.10', limite_linhas=13).carregar()
    display(amostra_original.iloc[:13, :8])
else:
    print('Arquivo bruto não incluído no pacote. As células oficiais estão registradas acima.')

3.10 - Número de Concluintes no Enem segundo a Dependência Administrativa por Região Geográfica e Unidade da Federação - 2020
Linha nacional no Excel: 10


,celula,valor_original
0,A10,Brasil
1,C10,1395827
2,D10,46594
3,E10,630939
4,F10,8331
5,G10,218705
6,H10,491258


Arquivo bruto não incluído no pacote. As células oficiais estão registradas acima.


## 4. Leitura das tabelas tratadas com Pandas

Cada base permanece separada. O vínculo entre séries para a dispersão é somente o ano, sem associação individual.
`rede_origem` representa a escola de origem; `categoria_ies` representa a instituição superior e está como “Todas” nesta seleção.

In [5]:
enem = FonteCSV(PROCESSED / 'enem_brasil_rede.csv').carregar()
superior = FonteCSV(PROCESSED / 'superior_brasil.csv').carregar()
basica = FonteCSV(PROCESSED / 'basica_matriculas_brasil.csv').carregar()
trajetoria = FonteCSV(PROCESSED / 'trajetoria_coorte_2020.csv').carregar()
amostra_trajetoria = FonteCSV(PROCESSED / 'trajetoria_amostra_25_linhas.csv').carregar()

for nome, tabela in [('Enem', enem), ('Superior', superior), ('Básica', basica), ('Trajetória', trajetoria)]:
    print(nome, '→', tabela.shape[0], 'linhas e', tabela.shape[1], 'colunas')
    display(tabela.head())

Enem → 15 linhas e 7 colunas


,ano,abrangencia,rede_origem,inscritos_concluintes,total_concluintes_inscritos,fonte_id,aba
0,2020,Brasil,Pública,685864,1395827,enem_2020,3.10
1,2020,Brasil,Privada,218705,1395827,enem_2020,3.10
2,2020,Brasil,Não dispõe da informação,491258,1395827,enem_2020,3.10
3,2021,Brasil,Pública,603041,1150857,enem_2021,3.10
4,2021,Brasil,Privada,210765,1150857,enem_2021,3.10


Superior → 5 linhas e 9 colunas


,ano,abrangencia,categoria_ies,modalidade,forma_ingresso,ingressantes_graduacao,ingressantes_universidades,fonte_id,aba
0,2020,Brasil,Todas,Presencial e a distância,Todas,3765475,2032571,superior_2020,4.40
1,2021,Brasil,Todas,Presencial e a distância,Todas,3944897,2138197,superior_2021,4.40
2,2022,Brasil,Todas,Presencial e a distância,Todas,4756728,2584923,superior_2022,4.40
3,2023,Brasil,Todas,Presencial e a distância,Todas,4993992,2624163,superior_2023,4.40
4,2024,Brasil,Todas,Presencial e a distância,Todas,5010433,2675687,superior_2024,4.40


Básica → 10 linhas e 7 colunas


,ano,abrangencia,rede_origem,matriculas_em_regular,universo,fonte_id,aba
0,2020,Brasil,Pública,6624804,Todas as séries do ensino médio regular; não s...,basica_2020,Ensino Médio 1.25
1,2020,Brasil,Privada,925949,Todas as séries do ensino médio regular; não s...,basica_2020,Ensino Médio 1.25
2,2021,Brasil,Pública,6835399,Todas as séries do ensino médio regular; não s...,basica_2021,Ensino Médio 1.25
3,2021,Brasil,Privada,935158,Todas as séries do ensino médio regular; não s...,basica_2021,Ensino Médio 1.25
4,2022,Brasil,Pública,6895219,Todas as séries do ensino médio regular; não s...,basica_2022,Ensino Médio 1.25


Trajetória → 5 linhas e 15 colunas


,ano_referencia,vinculos_ingressantes_coorte,vinculos_permanencia,conclusoes_no_ano,desistencias_no_ano,falecimentos_no_ano,ausentes_qt_ingressante,ausentes_qt_permanencia,ausentes_qt_concluinte,ausentes_qt_desistencia,ausentes_qt_falecido,cursos_observados,ano_ingresso,fonte_id,unidade
0,2020,3859448,3214243,42358,602677,170,0,0,0,0,0,34848,2020,trajetoria_2020_2024,Vínculos em cursos; não pessoas únicas
1,2021,3859448,2284306,151399,778112,426,0,0,0,0,0,34848,2020,trajetoria_2020_2024,Vínculos em cursos; não pessoas únicas
2,2022,3859448,1554937,206928,522238,203,0,0,0,0,0,34848,2020,trajetoria_2020_2024,Vínculos em cursos; não pessoas únicas
3,2023,3859448,1010329,243853,300419,336,0,0,0,0,0,34848,2020,trajetoria_2020_2024,Vínculos em cursos; não pessoas únicas
4,2024,3859448,522934,296288,191002,105,0,0,0,0,0,34848,2020,trajetoria_2020_2024,Vínculos em cursos; não pessoas únicas


## 5. Tipos, ausências, duplicidades e estatísticas

Mostramos as verificações pedidas na Etapa 2 do enunciado: `info()`, `isna().sum()`, `duplicated()` e `describe()`.
As estatísticas descrevem contagens anuais agregadas, não notas nem características individuais de estudantes.
“Não dispõe da informação” é uma categoria explícita; por isso não aparece como `NaN` em `rede_origem`.

In [6]:
bases = {'Enem': (enem, ['ano', 'rede_origem']),
         'Educação Superior': (superior, ['ano']),
         'Educação Básica': (basica, ['ano', 'rede_origem']),
         'Trajetória': (trajetoria, ['ano_ingresso', 'ano_referencia'])}
for nome, (tabela, chave) in bases.items():
    print()
    print('BASE:', nome)
    tabela.info()
    display(tabela.isna().sum().rename('valores_ausentes').to_frame())
    print('Linhas repetidas:', tabela.duplicated().sum())
    print('Chaves repetidas:', tabela.duplicated(chave).sum())
    display(tabela.select_dtypes('number').describe())

print('Amostra de trajetória: dados por curso, coorte e ano de referência')
amostra_trajetoria.info()
display(amostra_trajetoria.isna().sum().to_frame('valores_ausentes'))
display(amostra_trajetoria.head())
perfil_trajetoria = json.loads((PROCESSED / 'perfil_trajetoria.json').read_text(encoding='utf-8'))
print('Diagnóstico do arquivo completo de trajetória:', perfil_trajetoria['registros'], 'registros')
display(pd.DataFrame({'tipo': perfil_trajetoria['tipos'],
                      'ausencias_base_completa': perfil_trajetoria['ausencias']}))
print(perfil_trajetoria['nota'])


BASE: Enem
<class 'pandas.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 7 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   ano                          15 non-null     int64  
 1   abrangencia                  15 non-null     str    
 2   rede_origem                  15 non-null     str    
 3   inscritos_concluintes        15 non-null     int64  
 4   total_concluintes_inscritos  15 non-null     int64  
 5   fonte_id                     15 non-null     str    
 6   aba                          15 non-null     float64
dtypes: float64(1), int64(3), str(3)
memory usage: 972.0 bytes


,valores_ausentes
ano,0
abrangencia,0
rede_origem,0
inscritos_concluintes,0
total_concluintes_inscritos,0
fonte_id,0
aba,0


Linhas repetidas: 0
Chaves repetidas: 0


,ano,inscritos_concluintes,total_concluintes_inscritos,aba
count,15.00,15.00,15.00,15.00
mean,"2,022.00","458,867.33","1,376,602.00",3.10
std,1.46,"256,731.10","155,953.74",0.00
min,"2,020.00",519.00,"1,150,857.00",3.10
25%,"2,021.00","228,524.50","1,317,560.00",3.10
50%,"2,022.00","442,658.00","1,395,827.00",3.10
75%,"2,023.00","705,193.50","1,401,164.00",3.10
max,"2,024.00","841,213.00","1,617,602.00",3.10



BASE: Educação Superior
<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   ano                         5 non-null      int64  
 1   abrangencia                 5 non-null      str    
 2   categoria_ies               5 non-null      str    
 3   modalidade                  5 non-null      str    
 4   forma_ingresso              5 non-null      str    
 5   ingressantes_graduacao      5 non-null      int64  
 6   ingressantes_universidades  5 non-null      int64  
 7   fonte_id                    5 non-null      str    
 8   aba                         5 non-null      float64
dtypes: float64(1), int64(3), str(5)
memory usage: 492.0 bytes


,valores_ausentes
ano,0
abrangencia,0
categoria_ies,0
modalidade,0
forma_ingresso,0
ingressantes_graduacao,0
ingressantes_universidades,0
fonte_id,0
aba,0


... (12 outputs left out)

,CO_IES,NO_IES,TP_CATEGORIA_ADMINISTRATIVA,TP_ORGANIZACAO_ACADEMICA,CO_CURSO,NO_CURSO,CO_REGIAO,CO_UF,CO_MUNICIPIO,TP_GRAU_ACADEMICO,TP_MODALIDADE_ENSINO,CO_CINE_ROTULO,NO_CINE_ROTULO,CO_CINE_AREA_GERAL,NO_CINE_AREA_GERAL,NU_ANO_INGRESSO,NU_ANO_REFERENCIA,NU_PRAZO_INTEGRALIZACAO,NU_ANO_INTEGRALIZACAO,NU_PRAZO_ACOMPANHAMENTO,NU_ANO_MAXIMO_ACOMPANHAMENTO,QT_INGRESSANTE,QT_PERMANENCIA,QT_CONCLUINTE,QT_DESISTENCIA,QT_FALECIDO,TAP,TCA,TDA,TCAN,TADA
0,1,UNIVERSIDADE FEDERAL DE MATO GROSSO,1,1,1,DIREITO,5,51,5103403,1,1,0421D01,Direito,4,"Negócios, administração e direito",2020,2020,6,2025,10,2029,100,100,0,0,0,100.00,0.00,0.00,0.00,0.00
1,1,UNIVERSIDADE FEDERAL DE MATO GROSSO,1,1,1,DIREITO,5,51,5103403,1,1,0421D01,Direito,4,"Negócios, administração e direito",2020,2021,6,2025,10,2029,100,97,0,3,0,97.00,0.00,3.00,0.00,3.00
2,1,UNIVERSIDADE FEDERAL DE MATO GROSSO,1,1,1,DIREITO,5,51,5103403,1,1,0421D01,Direito,4,"Negócios, administração e direito",2020,2022,6,2025,10,2029,100,92,1,4,0,92.00,1.00,7.00,1.00,4.00
3,1,UNIVERSIDADE FEDERAL DE MATO GROSSO,1,1,1,DIREITO,5,51,5103403,1,1,0421D01,Direito,4,"Negócios, administração e direito",2020,2023,6,2025,10,2029,100,91,1,0,0,91.00,2.00,7.00,1.00,0.00
4,1,UNIVERSIDADE FEDERAL DE MATO GROSSO,1,1,1,DIREITO,5,51,5103403,1,1,0421D01,Direito,4,"Negócios, administração e direito",2020,2024,6,2025,10,2029,100,34,57,0,0,34.00,59.00,7.00,57.00,0.00


Diagnóstico do arquivo completo de trajetória: 174240 registros


,tipo,ausencias_base_completa
CO_IES,object,0
NO_IES,str,0
TP_CATEGORIA_ADMINISTRATIVA,object,0
TP_ORGANIZACAO_ACADEMICA,object,0
CO_CURSO,object,0
NO_CURSO,str,0
CO_REGIAO,object,26230
CO_UF,object,26230
CO_MUNICIPIO,object,26230
TP_GRAU_ACADEMICO,object,0


Ausências geográficas preservadas; agregação nacional inclui esses cursos.


## 6. Tratamento e reconciliação

- Preservamos os arquivos brutos. Apenas os CSVs agregados entram no Git.
- Selecionamos a linha Brasil, sem somar Brasil + regiões + estados.
- Conferimos a soma das dependências do Enem, das organizações acadêmicas no Censo Superior e das matrículas por série/dependência no Censo Escolar.
- Não excluímos linhas nem removemos duplicidades sem diagnóstico. Uma chave duplicada interrompe a preparação.
- Não transformamos vazios ou símbolos em zero. Ausências numéricas necessárias à reconciliação interrompem o processamento.
- Os dados de trajetória resumem a mesma coorte em cada referência; não somamos os ingressantes de 2020 novamente a cada ano.

In [7]:
conferencia = enem.groupby('ano').agg(
    soma_categorias=('inscritos_concluintes', 'sum'),
    total_publicado=('total_concluintes_inscritos', 'first'))
conferencia['confere'] = conferencia.soma_categorias.eq(conferencia.total_publicado)
display(conferencia)
assert conferencia.confere.all()
assert set(enem.ano) == set(range(2020, 2025))
assert not enem.duplicated(['ano', 'rede_origem']).any()
assert (superior.ingressantes_universidades <= superior.ingressantes_graduacao).all()
assert len(evidencias) == 15
print('Totais nacionais e chaves conferidos.')

,soma_categorias,total_publicado,confere
ano,,,
2020,1395827,1395827,True
2021,1150857,1150857,True
2022,1317560,1317560,True
2023,1401164,1401164,True
2024,1617602,1617602,True


Totais nacionais e chaves conferidos.


## 7. Indicadores exploratórios

Calculamos contagens de inscrições por rede, o total, a participação da rede pública **entre todos os concluintes inscritos**, a proporção sem informação e a variação anual da contagem pública.

**Fórmulas:** participação pública = inscrições públicas / total de concluintes inscritos × 100;
sem informação = inscrições sem informação / mesmo total × 100;
variação anual = (contagem pública atual / anterior − 1) × 100.

São indicadores descritivos. Não representam a proporção de todos os concluintes do Brasil que fizeram o Enem.
Não calculamos a taxa de transição para a graduação, pois ela exigiria identificar a coorte de egressos e seus ingressos no ano seguinte.

In [8]:
resumo = AnalisadorColeta(enem).resumo()
display(resumo)
print('O primeiro ano não tem variação anual calculável nesta janela:',
      pd.isna(resumo.loc[0, 'variacao_anual_publica_pct']))
ano_2024 = resumo.loc[resumo.ano.eq(2024)].iloc[0]
print(f'2024: privada = {ano_2024.inscritos_privada:,.0f}; '
      f'sem informação = {ano_2024.inscritos_sem_informacao:,.0f} '
      f'({ano_2024.sem_informacao_pct:.2f}% do total).')
print('A causa dessa mudança acentuada da categoria privada não foi confirmada.')

,ano,inscritos_sem_informacao,inscritos_privada,inscritos_publica,total_concluintes_inscritos,participacao_publica_no_total_pct,sem_informacao_pct,variacao_anual_publica_pct
0,2020,491258,218705,685864,1395827,49.14,35.19,NaN
1,2021,337051,210765,603041,1150857,52.40,29.29,-12.08
2,2022,365616,227421,724523,1317560,54.99,27.75,20.14
3,2023,442658,229628,728878,1401164,52.02,31.59,0.60
4,2024,775870,519,841213,1617602,52.00,47.96,15.41


O primeiro ano não tem variação anual calculável nesta janela: True
2024: privada = 519; sem informação = 775,870 (47.96% do total).
A causa dessa mudança acentuada da categoria privada não foi confirmada.


## 8. Quatro gráficos interativos

As figuras são produzidas pela classe `Visualizador`, que recebe tabelas prontas e devolve figuras Plotly.
Podem ser reutilizadas com `st.plotly_chart(figura)` pelo colega responsável pelo dashboard.

In [9]:
visualizador = Visualizador()
figuras = visualizador.todas(enem, superior)
# O formato Plotly é mantido na saída para ambientes compatíveis, como Jupyter.
# No GitHub, use também os HTMLs, pois a renderização do notebook pode ser estática.
for nome, figura in figuras.items():
    print(nome)
    display(figura)

01_linhas


02_barras


03_setores


04_dispersao


### Como interpretar

**Linhas:** a contagem pública cai em 2021 e cresce nas edições seguintes. A leitura exige considerar a cobertura da rede e as particularidades da pandemia; o gráfico sozinho não explica as causas.

**Barras:** comparam números absolutos das redes registradas. A queda acentuada da categoria privada em 2024 é um alerta de comparabilidade, não evidência suficiente de uma queda real da participação.

**Setores:** os registros sem informação da rede integram o total de 2024. Não recalculamos a composição omitindo essa categoria.

**Dispersão:** cada ponto representa um ano, comparando inscrições de concluintes públicos no Enem com ingressos de todas as origens e formas na graduação. Não há ligação entre pessoas, teste de correlação ou conclusão de causa e efeito.

## 9. Exportação dos gráficos

Além das figuras exibidas acima, a próxima célula salva os HTMLs em `data/processed/graficos/`.
Abra `index.html` dessa pasta para ver os quatro gráficos com explicações. Mantenha `plotly.min.js` junto dos HTMLs.
O código dessa exportação também está inteiro no notebook, sem dependência de scripts externos.

In [10]:
def gerar():
    processados = ROOT/'data/processed'
    enem = FonteCSV(processados/'enem_brasil_rede.csv').carregar()
    superior = FonteCSV(processados/'superior_brasil.csv').carregar()
    resumo = AnalisadorColeta(enem).resumo()
    resumo.to_csv(processados/'indicadores_exploratorios.csv', index=False, encoding='utf-8')
    visualizador = Visualizador()
    figuras = visualizador.todas(enem, superior)
    pasta = visualizador.exportar(figuras, ROOT/'data/processed/graficos')
    comentarios = [
        'A contagem da rede pública diminui em 2021 e aumenta nas edições seguintes. '
        'A série representa apenas a rede registrada na sinopse, com cobertura incompleta.',
        'São números absolutos, não probabilidades de acesso. Em 2024 a categoria privada '
        'contém somente 519 inscrições, enquanto 775.870 registros não dispõem de informação. '
        'Esses valores constam na fonte; não permitem concluir uma queda real da participação privada.',
        'O setor cinza integra o denominador. Não redistribuímos os registros sem informação '
        'entre as redes pública e privada.',
        'Cada ponto compara o mesmo ano de duas séries com universos distintos. Os ingressantes '
        'incluem todas as formas de ingresso, redes de origem, modalidades e organizações acadêmicas. '
        'Os pontos não acompanham indivíduos, não formam um funil e não comprovam causalidade.',
    ]
    secoes = []
    for (nome, figura), comentario in zip(figuras.items(), comentarios):
        secoes.append('<section>'+figura.to_html(full_html=False, include_plotlyjs=False,
                                               config={'displaylogo':False, 'responsive':True})+
                      '<p>'+comentario+'</p><a href="'+nome+'.html">Abrir gráfico separado</a></section>')
    pagina = '''<!doctype html><html lang="pt-BR"><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Do Enem à graduação — coleta e gráficos</title>
<script src="plotly.min.js"></script>
<style>body{margin:0;background:#f1f5f8;color:#23334a;font:16px/1.6 Arial,sans-serif}
main{max-width:1180px;margin:auto;padding:32px 24px}h1{line-height:1.2}section{background:white;
margin:26px 0;padding:18px;border-radius:12px}section p{margin:0 24px 12px}a{color:#086d81}
.aviso{background:#fff1d6;border-left:5px solid #d88a21;padding:14px 20px}</style>
<main><h1>Do Enem à graduação</h1><p>Brasil · 2020 a 2024 · exploração de dados oficiais do Inep</p>
<p class="aviso">Esta análise compara participação no Enem e ingresso na graduação.
Não identifica estudantes da rede pública que ingressaram pelo Enem.
Inscrição não é presença nem aprovação. A edição de 2020 foi aplicada em 2021.</p>
'''+''.join(secoes)+'''<p>Fontes: <a href="https://www.gov.br/inep/pt-br/acesso-a-informacao/dados-abertos/sinopses-estatisticas/enem">Sinopses do Enem</a> e
<a href="https://www.gov.br/inep/pt-br/acesso-a-informacao/dados-abertos/sinopses-estatisticas/educacao-superior">Sinopses da Educação Superior</a>.
Consulte os CSVs, o notebook e o catálogo de fontes para rastrear os valores.</p></main></html>'''
    (pasta/'index.html').write_text(pagina, encoding='utf-8')
    print('Quatro gráficos exportados. Abra data/processed/graficos/index.html no navegador.')
    return figuras

_ = gerar()
arquivos = sorted(p.name for p in (PROCESSED/'graficos').glob('*.html'))
print('Arquivos gerados:', arquivos)
assert len(arquivos) == 5
assert (PROCESSED/'graficos/plotly.min.js').exists()


Quatro gráficos exportados. Abra data/processed/graficos/index.html no navegador.
Arquivos gerados: ['01_linhas.html', '02_barras.html', '03_setores.html', '04_dispersao.html', 'index.html']


## 10. Minha entrega ao grupo

- `data/processed/`: tabelas nacionais, catálogo de fontes, evidências, diagnóstico e gráficos exportados.
- `data/raw/`: arquivos originais do Inep apenas no computador; o `.gitignore` dessa própria pasta impede o envio dos brutos.
- `notebooks/01_exploracao.ipynb`: exploração executada, código e quatro gráficos interativos.
- `notebooks/LEIA_PRIMEIRO.md`: instruções de execução e de envio destas pastas.

Esta contribuição não altera `src/`, `scripts/`, `tests/`, `README.md`, `requirements.txt` nem o `.gitignore` da raiz.
O dashboard, a integração das classes na aplicação e a documentação geral continuam com as outras frentes.

As conclusões permitidas e as limitações são as mesmas: não se mede aprovação nem ingresso individual pelo Enem,
e a categoria privada de 2024 exige cautela devido à alteração acentuada dos registros na fonte.

As fontes oficiais, links, abas e hashes estão em `data/processed/fontes.json`.
Os caminhos `*_na_coleta_original` documentam a organização anterior da coleta;
`arquivo_zip` e `arquivo_xlsx` indicam a localização esperada nesta entrega.

## Apêndice opcional: repetir a coleta e o tratamento

As tabelas já foram preparadas e validadas. **Não é necessário rodar o download para usar esta entrega.**
O código abaixo permite reproduzir a preparação sem arquivos em `src/` ou `scripts/`.
Ele lê os ZIPs de `data/raw/` e escreve apenas em `data/processed/`.

Para refazer, instale `truststore`, execute as duas células do apêndice com os controles ativados e depois
execute novamente as células de leitura, análise e gráficos acima. Os ZIPs oficiais somam alguns GB.
Cabeçalhos e totais são verificados antes de gerar os CSVs; valores desconhecidos não viram zero.

In [11]:
"""Extração auditável de totais nacionais; nenhuma soma de subtotais geográficos."""
from pathlib import Path
import numbers
import unicodedata

import pandas as pd
from openpyxl.utils import get_column_letter



def normalizar(valor):
    texto = unicodedata.normalize('NFKD', str(valor))
    return ' '.join(''.join(c for c in texto if not unicodedata.combining(c)).lower().split())


def contagem(valor):
    """Não inventa zero para '-', '...', célula vazia ou símbolo desconhecido."""
    if pd.isna(valor) or str(valor).strip() in {'', '-', '...', '…'}:
        return pd.NA
    if not isinstance(valor, numbers.Real) or valor < 0 or int(valor) != valor:
        raise ValueError(f'Contagem inesperada: {valor!r}')
    return int(valor)


def soma_completa(valores):
    valores = list(valores)
    return pd.NA if any(pd.isna(v) for v in valores) else sum(valores)


def conferir_soma(partes, total, contexto):
    calculado = soma_completa(partes)
    if pd.isna(calculado) or pd.isna(total) or calculado != total:
        raise ValueError(f'Total não reconciliado em {contexto}: {calculado} != {total}')


class ExtratorSinopse:
    def __init__(self):
        self._evidencias = []

    @property
    def evidencias(self):
        return list(self._evidencias)

    def nacional(self, arquivo, aba, ano, termos_titulo):
        tabela = FonteExcel(arquivo, aba, limite_linhas=24).carregar()
        titulo = ' '.join(str(v) for v in tabela.iloc[3] if pd.notna(v))
        if str(ano) not in titulo or any(normalizar(t) not in normalizar(titulo) for t in termos_titulo):
            raise ValueError(f'Título inesperado: {arquivo.name}/{aba}: {titulo}')
        linhas = tabela.index[tabela[0].map(normalizar).isin(['brasil', 'total brasil'])]
        if len(linhas) != 1:
            raise ValueError(f'Esperada uma linha Brasil em {arquivo.name}/{aba}')
        indice = int(linhas[0])
        self._evidencias.append({
            'ano': ano, 'arquivo': arquivo.name, 'aba': aba, 'titulo': titulo,
            'linha_excel': indice + 1,
            'cabecalho': [[None if pd.isna(v) else v for v in row]
                         for row in tabela.iloc[5:indice, :min(tabela.shape[1], 100)].values.tolist()],
            'celulas': {f'{get_column_letter(i+1)}{indice+1}': v
                        for i, v in enumerate(tabela.iloc[indice])
                        if pd.notna(v) and i < 100},
        })
        return tabela, tabela.iloc[indice]

    def enem(self, arquivo, ano):
        tabela, brasil = self.nacional(arquivo, '3.10', ano, ['concluintes', 'dependência'])
        # O layout é conferido em C:H antes de acessar os valores.
        esperado = ['Federal', 'Estadual', 'Municipal', 'Privada']
        if [normalizar(v) for v in tabela.iloc[7, 3:7]] != [normalizar(v) for v in esperado]:
            raise ValueError(f'Layout do Enem mudou em {ano}')
        if 'informacao' not in normalizar(tabela.iloc[6, 7]):
            raise ValueError('Categoria sem informação não identificada')
        total = contagem(brasil.iloc[2])
        federal, estadual, municipal, privada, ignorada = map(contagem, brasil.iloc[3:8])
        publica = soma_completa([federal, estadual, municipal])
        conferir_soma([publica, privada, ignorada], total, f'Enem {ano}')
        return [dict(ano=ano, abrangencia='Brasil', rede_origem=rede,
                     inscritos_concluintes=valor, total_concluintes_inscritos=total,
                     fonte_id=f'enem_{ano}', aba='3.10')
                for rede, valor in [('Pública', publica), ('Privada', privada),
                                    ('Não dispõe da informação', ignorada)]]

    def superior(self, arquivo, ano):
        tabela, brasil = self.nacional(arquivo, '4.4', ano, ['ingressantes', 'graduação'])
        if normalizar(tabela.iloc[6, 4]) != 'total' or normalizar(tabela.iloc[6, 5]) != 'universidade':
            raise ValueError(f'Layout da Educação Superior mudou em {ano}')
        # E=graduação em todas as IES; F=universidades. Não são a mesma população.
        total, universidades = map(contagem, brasil.iloc[4:6])
        # F, AI, BL, CO: organizações acadêmicas mutuamente exclusivas.
        conferir_soma([contagem(brasil.iloc[i]) for i in [5, 34, 63, 92]], total,
                      f'Educação Superior {ano}')
        return dict(ano=ano, abrangencia='Brasil', categoria_ies='Todas',
                    modalidade='Presencial e a distância', forma_ingresso='Todas',
                    ingressantes_graduacao=total, ingressantes_universidades=universidades,
                    fonte_id=f'superior_{ano}', aba='4.4')

    def basica(self, arquivo, ano):
        aba = 'Ensino Médio 1.25'
        tabela, brasil = self.nacional(arquivo, aba, ano, ['matrículas', 'ensino médio'])
        if [normalizar(v) for v in tabela.iloc[8, 6:10]] != ['federal', 'estadual', 'municipal', 'privada']:
            raise ValueError(f'Layout da Educação Básica mudou em {ano}')
        total = contagem(brasil.iloc[4])
        # Cada série contém quatro dependências. Não somar as colunas Total.
        dependencias = [soma_completa(contagem(brasil.iloc[i]) for i in range(col, 30, 5))
                        for col in range(6, 10)]
        conferir_soma(dependencias, total, f'Educação Básica {ano}')
        return [dict(ano=ano, abrangencia='Brasil', rede_origem=rede,
                     matriculas_em_regular=valor,
                     universo='Todas as séries do ensino médio regular; não são concluintes',
                     fonte_id=f'basica_{ano}', aba=aba)
                for rede, valor in [('Pública', soma_completa(dependencias[:3])),
                                    ('Privada', dependencias[3])]]


def tabela_trajetoria(arquivo):
    """Resume cobertura e fluxos ANUAIS da mesma coorte, sem média de taxas."""
    tabela = FonteExcel(arquivo, 'INDICADORES_TRAJETORIA').carregar()
    cabecalho = tabela.iloc[8].tolist()
    df = tabela.iloc[9:].copy()
    df.columns = cabecalho
    df = df[df['CO_IES'].notna()].copy()
    # A última linha contém a referência bibliográfica em CO_IES, não um curso.
    rodape = df['CO_IES'].map(normalizar).eq('fonte: censo da educacao superior/inep.')
    if rodape.sum() > 1 or df.loc[rodape, 'CO_CURSO'].notna().any():
        raise ValueError('Rodapé de trajetória inesperado')
    df = df.loc[~rodape].copy()
    chave = ['CO_IES', 'CO_CURSO', 'NU_ANO_INGRESSO', 'NU_ANO_REFERENCIA']
    if df.duplicated(chave).any():
        raise ValueError('Duplicidades na chave curso/coorte/ano; investigar antes de agregar')
    numericas = ['NU_ANO_INGRESSO', 'NU_ANO_REFERENCIA', 'QT_INGRESSANTE',
                 'QT_PERMANENCIA', 'QT_CONCLUINTE', 'QT_DESISTENCIA', 'QT_FALECIDO']
    for col in numericas:
        df[col] = pd.to_numeric(df[col], errors='raise').astype('Int64')
    if set(df['NU_ANO_INGRESSO'].dropna()) != {2020}:
        raise ValueError('Coorte diferente de 2020')
    if df[numericas[:2]].isna().any().any():
        raise ValueError('Ano de ingresso/referência ausente na trajetória')
    grupos = df.groupby('NU_ANO_REFERENCIA')
    # Há contagens ausentes no arquivo oficial. Um total incompleto fica NA,
    # acompanhado da quantidade de registros sem contagem, nunca vira zero.
    soma = grupos[numericas[2:]].agg(lambda serie: serie.sum(min_count=len(serie)))
    for coluna in numericas[2:]:
        soma['ausentes_' + coluna.lower()] = grupos[coluna].apply(lambda serie: int(serie.isna().sum()))
    soma['cursos_observados'] = df.groupby('NU_ANO_REFERENCIA').size()
    soma = soma.reset_index().rename(columns={
        'NU_ANO_REFERENCIA':'ano_referencia', 'QT_INGRESSANTE':'vinculos_ingressantes_coorte',
        'QT_PERMANENCIA':'vinculos_permanencia', 'QT_CONCLUINTE':'conclusoes_no_ano',
        'QT_DESISTENCIA':'desistencias_no_ano', 'QT_FALECIDO':'falecimentos_no_ano'})
    soma['ano_ingresso'] = 2020
    soma['fonte_id'] = 'trajetoria_2020_2024'
    soma['unidade'] = 'Vínculos em cursos; não pessoas únicas'
    amostra = df.head(25)
    perfil = dict(registros=int(len(df)), cursos=int(df[['CO_IES','CO_CURSO']].drop_duplicates().shape[0]),
                  linhas_de_rodape=int(rodape.sum()), duplicidades_chave=int(df.duplicated(chave).sum()),
                  tipos={c:str(t) for c,t in df.dtypes.items()},
                  ausencias={c:int(v) for c,v in df.isna().sum().items()},
                  nota='Ausências geográficas preservadas; agregação nacional inclui esses cursos.')
    return soma, amostra, perfil

import hashlib
import os
import shutil
import ssl
import subprocess
import sys
import zipfile
from datetime import datetime
from urllib.request import urlopen

PAGINA = 'https://www.gov.br/inep/pt-br/acesso-a-informacao/dados-abertos/'
DOWNLOAD = 'https://download.inep.gov.br/'

def fontes_oficiais():
    fontes = []
    for ano in range(2020, 2025):
        nome_enem = 'Sinopse_Enem2020.zip' if ano == 2020 else f'sinopse_enem_{ano}.zip'
        prefixo_basica = 'sinopses_estatisticas' if ano <= 2022 else 'sinopse_estatistica'
        for base, pagina, url, aba in [
            ('enem', 'sinopses-estatisticas/enem',
             f'informacoes_estatisticas/sinopses_estatisticas/sinopses_enem/{ano}/{nome_enem}', '3.10'),
            ('superior', 'sinopses-estatisticas/educacao-superior',
             f'informacoes_estatisticas/sinopses_estatisticas/sinopses_educacao_superior/sinopse_educacao_superior_{ano}.zip', '4.4'),
            ('basica', 'sinopses-estatisticas/educacao-basica',
             f'dados_abertos/sinopses_estatisticas/{prefixo_basica}_censo_escolar_{ano}.zip', 'Ensino Médio 1.25'),
        ]:
            fontes.append(dict(id=f'{base}_{ano}', base=base, ano=ano,
                               pagina=PAGINA+pagina, url=DOWNLOAD+url, aba=aba))
    fontes.append(dict(id='trajetoria_2020_2024', base='trajetoria', ano='2020-2024',
                       pagina=PAGINA+'indicadores-educacionais/indicadores-de-trajetoria-da-educacao-superior',
                       url=DOWNLOAD+'informacoes_estatisticas/indicadores_educacionais/indicadores_trajetoria_es_2020_2024.zip',
                       aba='INDICADORES_TRAJETORIA'))
    return fontes

def baixar_zip(url, destino):
    # Usa autoridades certificadoras do sistema, mantendo validação TLS ativa.
    import truststore
    contexto = truststore.SSLContext(ssl.PROTOCOL_TLS_CLIENT)
    temporario = destino.with_suffix('.part')
    if sys.platform == 'win32':
        # O Windows pode precisar da pilha HTTP do sistema em redes corporativas.
        # URL e caminho são dados de ambiente, sem interpolar texto no comando.
        ambiente = {**os.environ, 'INEP_COLETA_URL':url, 'INEP_COLETA_DESTINO':str(temporario)}
        subprocess.run(['powershell.exe', '-NoProfile', '-NonInteractive', '-Command',
                        "$ProgressPreference='SilentlyContinue'; "
                        'Invoke-WebRequest -UseBasicParsing -Uri $env:INEP_COLETA_URL '
                        '-OutFile $env:INEP_COLETA_DESTINO -TimeoutSec 60 -ErrorAction Stop'],
                       env=ambiente, check=True, creationflags=subprocess.CREATE_NO_WINDOW)
    else:
        with urlopen(url, context=contexto, timeout=60) as resposta, temporario.open('wb') as saida:
            shutil.copyfileobj(resposta, saida)
    if not zipfile.is_zipfile(temporario):
        raise ValueError(f'O endereço não retornou ZIP válido: {url}')
    temporario.replace(destino)

def localizar_planilha(fonte, baixar=False):
    base, ano = fonte['base'], fonte['ano']
    pasta = ROOT/'data/raw'/base
    if base != 'trajetoria':
        pasta /= str(ano)
    pasta.mkdir(parents=True, exist_ok=True)
    zips = list(pasta.glob('*.zip'))
    if not zips and baixar:
        destino = pasta/Path(fonte['url']).name
        print(f'Baixando {fonte["id"]} (os ZIPs podem ter centenas de MB)...', flush=True)
        baixar_zip(fonte['url'], destino)
        zips = [destino]
    if len(zips) != 1:
        raise FileNotFoundError(f'Esperado um ZIP em {pasta}. Use --baixar ou baixe {fonte["url"]}')
    zip_path = zips[0]
    with zipfile.ZipFile(zip_path) as arquivo:
        nomes = [n for n in arquivo.namelist() if n.lower().endswith('.xlsx')
                 and 'anexo' not in n.lower() and not Path(n).name.startswith('~$')]
        if len(nomes) != 1:
            raise ValueError(f'Planilha principal ambígua em {zip_path.name}: {nomes}')
        # Apenas a planilha principal é necessária. Não extrair anexos enormes.
        planilha = pasta/Path(nomes[0]).name
        # Reextrair para garantir que a tabela lida corresponde ao ZIP auditado.
        with arquivo.open(nomes[0]) as entrada, planilha.open('wb') as saida:
            shutil.copyfileobj(entrada, saida)
        fonte['membro_zip'] = nomes[0]
    with zip_path.open('rb') as entrada:
        fonte['sha256_zip'] = hashlib.file_digest(entrada, 'sha256').hexdigest()
    fonte['sha256_xlsx'] = hashlib.sha256(planilha.read_bytes()).hexdigest()
    fonte['arquivo_zip'] = zip_path.relative_to(ROOT).as_posix()
    fonte['arquivo_xlsx'] = planilha.relative_to(ROOT).as_posix()
    fonte['data_arquivo_local'] = datetime.fromtimestamp(zip_path.stat().st_mtime).astimezone().isoformat()
    fonte['data_processamento'] = datetime.now().astimezone().isoformat()
    return planilha

def preparar(baixar=False):
    fontes = fontes_oficiais()
    extrator = ExtratorSinopse()
    registros = {'enem': [], 'superior': [], 'basica': []}
    saida = ROOT/'data/processed'
    saida.mkdir(parents=True, exist_ok=True)
    for fonte in fontes:
        print('Processando', fonte['id'], flush=True)
        arquivo = localizar_planilha(fonte, baixar)
        base = fonte['base']
        if base == 'trajetoria':
            trajetoria, amostra, perfil_trajetoria = tabela_trajetoria(arquivo)
        else:
            resultado = getattr(extrator, base)(arquivo, fonte['ano'])
            registros[base].extend(resultado if isinstance(resultado, list) else [resultado])
    tabelas = {base: pd.DataFrame(linhas) for base, linhas in registros.items()}
    for base, df in tabelas.items():
        chave = ['ano'] + (['rede_origem'] if base in ['enem', 'basica'] else [])
        if df.duplicated(chave).any():
            raise ValueError(f'Chave duplicada em {base}')
        if set(df['ano']) != set(range(2020, 2025)):
            raise ValueError(f'Anos incompletos em {base}')
    nomes = {'enem':'enem_brasil_rede.csv', 'superior':'superior_brasil.csv',
             'basica':'basica_matriculas_brasil.csv'}
    for base, df in tabelas.items():
        df.to_csv(saida/nomes[base], index=False, encoding='utf-8')
    trajetoria.to_csv(saida/'trajetoria_coorte_2020.csv', index=False, encoding='utf-8')
    amostra.to_csv(saida/'trajetoria_amostra_25_linhas.csv', index=False, encoding='utf-8')
    (saida/'perfil_trajetoria.json').write_text(
        json.dumps(perfil_trajetoria, ensure_ascii=False, indent=2), encoding='utf-8')
    (saida/'fontes.json').write_text(json.dumps(fontes, ensure_ascii=False, indent=2), encoding='utf-8')
    (saida/'evidencias_extracao.json').write_text(
        json.dumps(extrator.evidencias, ensure_ascii=False, indent=2), encoding='utf-8')
    print('Pronto: tabelas nacionais e evidências em dados/processed.', flush=True)

In [12]:
REPROCESSAR_FONTES = False
BAIXAR_FONTES = False

if REPROCESSAR_FONTES:
    preparar(baixar=BAIXAR_FONTES)
    print('Preparação concluída. Reexecute as células de leitura, análise e gráficos.')
else:
    print('Coleta opcional desativada. Usamos os CSVs oficiais tratados que acompanham esta entrega.')

Coleta opcional desativada. Usamos os CSVs oficiais tratados que acompanham esta entrega.
